# LeetCode 482 — License Key Formatting bằng Python

**Link đề:** https://leetcode.com/problems/license-key-formatting/
**Độ khó:** Easy · **Chủ đề:** String (xử lý chuỗi)

```python
class Solution:
    def licenseKeyFormatting(self, s: str, k: int) -> str:
        ...
```

## Mục tiêu học
1. Hiểu đề: bỏ hết dấu `-` cũ, viết hoa, rồi chia lại thành các nhóm `k` ký tự, **nhóm đầu được phép ngắn hơn**.
2. Ôn các công cụ chuỗi: `replace`, `upper`, slicing `s[a:b]`, `range(start, stop, step)`, `'-'.join(list)`, `reversed`, `%`.
3. Đi từ cách "tính độ dài nhóm đầu rồi cắt" → duyệt **ngược** chèn dấu `-` → hiểu vì sao cộng chuỗi trong vòng lặp có thể chậm.
4. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, xử lý ca biên (chuỗi toàn dấu `-`).
5. Làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi license key `s` gồm chữ cái, chữ số và dấu `-`, cùng số nguyên `k`. Định dạng lại sao cho:
- Mỗi nhóm có **đúng `k` ký tự**, **trừ nhóm đầu tiên** có thể ngắn hơn `k` nhưng phải có ít nhất 1 ký tự.
- Giữa hai nhóm có đúng một dấu `-`.
- Mọi chữ thường đổi thành **chữ hoa**.

| `s` | `k` | Output | Giải thích |
|---|:---:|---|---|
| `"5F3Z-2e-9-w"` | 4 | `"5F3Z-2E9W"` | 8 ký tự → 2 nhóm 4 |
| `"2-5g-3-J"` | 2 | `"2-5G-3J"` | 5 ký tự → nhóm đầu 1, rồi 2, 2 |
| `"---"` | 3 | `""` | không còn ký tự nào |

**Constraints:** `1 <= s.length <= 10^5`; `s` gồm chữ cái tiếng Anh, chữ số và `-`; `1 <= k <= 10^4`.

Nhận xét then chốt: các nhóm **cũ** trong `s` không còn ý nghĩa gì — chỉ thứ tự các ký tự (không phải `-`) là quan trọng. Và vì **nhóm đầu** là nhóm "lẻ", việc chia nhóm nên tính **từ cuối** lên.

In [1]:
s = "2-5g-3-J"
chars = s.replace("-", "").upper()
print("ký tự còn lại:", chars, "| số ký tự:", len(chars))
print("chia từ cuối theo k = 2:", "2 | 5G | 3J")

ký tự còn lại: 25G3J | số ký tự: 5
chia từ cuối theo k = 2: 2 | 5G | 3J


Câu hỏi đặt ra trước khi code:
- Nhóm đầu dài bao nhiêu? (gợi ý: phần dư khi chia số ký tự cho `k`)
- Nếu số ký tự chia hết cho `k` thì nhóm đầu dài bao nhiêu? (không được là 0!)
- Ghép chuỗi thế nào cho nhanh khi `s` dài `10^5`?

## 2. Nền tảng Python cần biết

### 2.1. `str.replace` và `str.upper`

- `s.replace(old, new)` trả **chuỗi mới** thay mọi `old` bằng `new`. Thay bằng `""` nghĩa là **xóa**.
- `s.upper()` trả chuỗi mới viết hoa; chữ số và ký tự khác giữ nguyên.

Chuỗi trong Python là **bất biến** (immutable): các phương thức không sửa `s` mà trả chuỗi mới.

In [2]:
s = "5F3Z-2e-9-w"
print(s.replace("-", ""))
print(s.upper())
print(s.replace("-", "").upper())
print("s gốc không đổi:", s)

5F3Z2e9w
5F3Z-2E-9-W
5F3Z2E9W
s gốc không đổi: 5F3Z-2e-9-w


### 2.2. `'-'.join(list_of_str)` — nối các chuỗi bằng dấu phân cách

`sep.join(items)` nối các phần tử (phải là chuỗi) và chèn `sep` **giữa** chúng (không chèn ở đầu/cuối). Với list rỗng → chuỗi rỗng.

In [3]:
print("-".join(["5F3Z", "2E9W"]))
print("-".join(["2", "5G", "3J"]))
print(repr("-".join([])))       # ''
print("-".join(["ABC"]))       # một phần tử: không có dấu -

5F3Z-2E9W
2-5G-3J
''
ABC


### 2.3. `range(start, stop, step)` và slicing để chia nhóm

`range(0, 10, 4)` sinh `0, 4, 8`. Dùng làm điểm bắt đầu mỗi nhóm: `t[i:i+k]`. Slicing tự "cắt bớt" nếu `i + k` vượt độ dài, không báo lỗi.

In [4]:
t = "ABCDEFGHIJ"
k = 4
print(list(range(0, len(t), k)))
print([t[i:i + k] for i in range(0, len(t), k)])   # nhóm cuối ngắn: 'IJ'
print(t[8:100])   # vượt độ dài vẫn an toàn

[0, 4, 8]
['ABCD', 'EFGH', 'IJ']
IJ


Chú ý: chia **từ đầu** như trên làm nhóm **cuối** bị ngắn — ngược với đề (nhóm **đầu** mới được ngắn). Ta sẽ sửa bằng cách tính độ dài nhóm đầu trước.

### 2.4. Phép `%` — độ dài nhóm đầu

Gọi `n` = số ký tự sau khi làm sạch. Nhóm đầu dài `n % k`; nếu `n % k == 0` thì nhóm đầu dài **đúng `k`** (các nhóm đều đủ). Viết gọn: `first = n % k or k`.

`a or b` trả `a` nếu `a` "truthy" (khác 0), ngược lại trả `b`.

In [5]:
for n, k in [(8, 4), (5, 2), (7, 3), (6, 3), (0, 3)]:
    first = n % k or k
    print(f"n={n}, k={k}: n % k = {n % k} -> nhóm đầu dài {first}")

n=8, k=4: n % k = 0 -> nhóm đầu dài 4
n=5, k=2: n % k = 1 -> nhóm đầu dài 1
n=7, k=3: n % k = 1 -> nhóm đầu dài 1
n=6, k=3: n % k = 0 -> nhóm đầu dài 3
n=0, k=3: n % k = 0 -> nhóm đầu dài 3


Với `n = 0` công thức cho `first = k`, nhưng `t[:k]` của chuỗi rỗng là `""` — ta sẽ xử lý ca này riêng cho rõ ràng.

## 3. Cách 1 — Trực giác: làm sạch, tính nhóm đầu, cắt phần còn lại

1. `t = s.replace('-', '').upper()`.
2. Nếu `t` rỗng → trả `""`.
3. `first = len(t) % k or k`; nhóm đầu là `t[:first]`.
4. Phần còn lại `t[first:]` có độ dài chia hết cho `k` → cắt đều thành các nhóm `k`.
5. `'-'.join(...)`.

In [6]:
def license_slice(s: str, k: int) -> str:
    t = s.replace("-", "").upper()
    if not t:
        return ""
    first = len(t) % k or k
    groups = [t[:first]]
    for i in range(first, len(t), k):
        groups.append(t[i:i + k])
    return "-".join(groups)

print(license_slice("5F3Z-2e-9-w", 4))   # 5F3Z-2E9W
print(license_slice("2-5g-3-J", 2))      # 2-5G-3J
print(repr(license_slice("---", 3)))     # ''

5F3Z-2E9W
2-5G-3J
''


`if not t:` — chuỗi rỗng là "falsy", nên `not t` là `True` khi `t == ""`.

### 3.1. TRACE cách cắt với `s = "2-5g-3-J"`, `k = 2`

| bước | biến | giá trị |
|---|---|---|
| làm sạch | `t` | `"25G3J"` (n = 5) |
| nhóm đầu | `first = 5 % 2 or 2` | 1 → `"2"` |
| i = 1 | `t[1:3]` | `"5G"` |
| i = 3 | `t[3:5]` | `"3J"` |
| ghép | `'-'.join` | `"2-5G-3J"` |

In [7]:
def trace_license_slice(s, k):
    t = s.replace("-", "").upper()
    print(f"làm sạch: {s!r} -> {t!r} (n = {len(t)})")
    if not t:
        print("rỗng -> ''")
        return ""
    first = len(t) % k or k
    groups = [t[:first]]
    print(f"nhóm đầu dài {first}: {t[:first]!r}")
    for i in range(first, len(t), k):
        groups.append(t[i:i + k])
        print(f"  i={i}: thêm {t[i:i + k]!r} -> groups = {groups}")
    result = "-".join(groups)
    print("kết quả:", repr(result))
    return result

trace_license_slice("2-5g-3-J", 2)
print()
trace_license_slice("a-a-a-a-", 1)

làm sạch: '2-5g-3-J' -> '25G3J' (n = 5)
nhóm đầu dài 1: '2'
  i=1: thêm '5G' -> groups = ['2', '5G']
  i=3: thêm '3J' -> groups = ['2', '5G', '3J']
kết quả: '2-5G-3J'

làm sạch: 'a-a-a-a-' -> 'AAAA' (n = 4)
nhóm đầu dài 1: 'A'
  i=1: thêm 'A' -> groups = ['A', 'A']
  i=2: thêm 'A' -> groups = ['A', 'A', 'A']
  i=3: thêm 'A' -> groups = ['A', 'A', 'A', 'A']
kết quả: 'A-A-A-A'


'A-A-A-A'

## 4. Nút thắt của cách "cộng chuỗi" và vì sao dùng `join`

Một cách viết rất phổ biến của người mới là cộng dần: `result = result + ch`. Vì chuỗi bất biến, **mỗi lần cộng có thể tạo chuỗi mới** và chép lại toàn bộ → về lý thuyết `O(n²)` với `n = 10^5`. (CPython có tối ưu riêng cho `+=` trên chuỗi trong nhiều trường hợp, nhưng **không được bảo đảm** — ví dụ khi còn biến khác tham chiếu tới chuỗi, hoặc trên hiện thực Python khác.)

Cách an toàn: gom các mảnh vào **list** (`append` trung bình `O(1)`) rồi `''.join` **một lần** ở cuối — `O(n)`.

Thí nghiệm nhỏ dưới đây cố tình giữ một tham chiếu phụ để tắt tối ưu, cho thấy cộng chuỗi chậm dần:

In [8]:
import time

def build_by_concat(n):
    res = ""
    for _ in range(n):
        keep = res          # tham chiếu phụ -> CPython buộc phải chép chuỗi
        res = res + "A"
    return res

def build_by_join(n):
    parts = []
    for _ in range(n):
        parts.append("A")
    return "".join(parts)

for n in [5000, 20000, 40000]:
    t0 = time.perf_counter(); build_by_concat(n); t1 = time.perf_counter()
    build_by_join(n); t2 = time.perf_counter()
    print(f"n={n:>6}: cộng chuỗi {t1 - t0:.4f}s | list + join {t2 - t1:.4f}s")

n=  5000: cộng chuỗi 0.0005s | list + join 0.0001s
n= 20000: cộng chuỗi 0.0032s | list + join 0.0006s
n= 40000: cộng chuỗi 0.0186s | list + join 0.0010s


(Số đo phụ thuộc máy; điều cần quan sát là cột "cộng chuỗi" tăng nhanh hơn tuyến tính.)

## 5. Insight — duyệt từ cuối, chèn `-` sau mỗi `k` ký tự

Vì nhóm "lẻ" nằm ở **đầu**, ta có thể đi **ngược** từ cuối chuỗi: đếm số ký tự đã lấy, cứ đủ `k` thì chèn `-` **trước khi** thêm ký tự kế tiếp. Cuối cùng đảo ngược kết quả. Cách này không cần tính `n % k` và chỉ duyệt `s` **một lần**, bỏ qua `-` ngay khi gặp (không cần tạo chuỗi sạch trước).

Công cụ: `reversed(s)` trả iterator duyệt ngược; `parts[::-1]` đảo list.

In [9]:
print(list(reversed("abc")))
print("".join(reversed("abc")))
print(["x", "y", "z"][::-1])

['c', 'b', 'a']
cba
['z', 'y', 'x']


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép

### 6.1. Đoạn 1: duyệt ngược, bỏ `-`
```python
for ch in reversed(s):
    if ch == '-':
        continue
```
### 6.2. Đoạn 2: đủ `k` ký tự trong nhóm hiện tại thì chèn `-` trước
```python
    if count == k:
        out.append('-')
        count = 0
    out.append(ch.upper())
    count += 1
```
Chèn `-` **trước** khi thêm ký tự mới (chứ không phải sau khi đủ nhóm) để không bao giờ tạo dấu `-` thừa ở cuối (tức là ở đầu chuỗi kết quả).
### 6.3. Đoạn 3: đảo ngược và ghép
```python
return ''.join(reversed(out))
```

In [10]:
def license_reverse(s: str, k: int) -> str:
    out = []
    count = 0                     # số ký tự trong nhóm đang xây (tính từ cuối)
    for ch in reversed(s):
        if ch == "-":
            continue
        if count == k:
            out.append("-")
            count = 0
        out.append(ch.upper())
        count += 1
    return "".join(reversed(out))

print(license_reverse("5F3Z-2e-9-w", 4))
print(license_reverse("2-5g-3-J", 2))
print(repr(license_reverse("---", 3)))

5F3Z-2E9W
2-5G-3J
''


## 7. TRACE từng bước — `s = "2-5g-3-J"`, `k = 2`

Duyệt ngược: `J, -, 3, -, g, 5, -, 2`.

| ch | bỏ qua? | `count` trước | chèn `-`? | `out` sau | `count` sau |
|---|---|---|---|---|---|
| J | | 0 | | `[J]` | 1 |
| - | ✔ | | | | |
| 3 | | 1 | | `[J,3]` | 2 |
| - | ✔ | | | | |
| g | | 2 | ✔ | `[J,3,-,G]` | 1 |
| 5 | | 1 | | `[J,3,-,G,5]` | 2 |
| - | ✔ | | | | |
| 2 | | 2 | ✔ | `[J,3,-,G,5,-,2]` | 1 |

Đảo ngược: `2-5G-3J` ✔.

In [11]:
def trace_license_reverse(s, k):
    out, count = [], 0
    for ch in reversed(s):
        if ch == "-":
            print(f"ch='-': bỏ qua")
            continue
        inserted = False
        if count == k:
            out.append("-")
            count = 0
            inserted = True
        out.append(ch.upper())
        count += 1
        print(f"ch={ch!r}: {'chèn - rồi ' if inserted else ''}thêm {ch.upper()!r} -> out={''.join(out)!r}, count={count}")
    result = "".join(reversed(out))
    print("đảo ngược ->", repr(result))
    return result

trace_license_reverse("2-5g-3-J", 2)

ch='J': thêm 'J' -> out='J', count=1
ch='-': bỏ qua
ch='3': thêm '3' -> out='J3', count=2
ch='-': bỏ qua
ch='g': chèn - rồi thêm 'G' -> out='J3-G', count=1
ch='5': thêm '5' -> out='J3-G5', count=2
ch='-': bỏ qua
ch='2': chèn - rồi thêm '2' -> out='J3-G5-2', count=1
đảo ngược -> '2-5G-3J'


'2-5G-3J'

## 8. Cách 3 (gọn) — cắt từ cuối bằng slicing

Làm sạch thành `t`, rồi lấy các nhóm **từ cuối**: điểm kết thúc `end = len(t), len(t)-k, ...`, nhóm là `t[max(0, end-k):end]`. Gom vào list, đảo lại, `join`. Cũng `O(n)`.

In [12]:
def license_slice_from_end(s: str, k: int) -> str:
    t = s.replace("-", "").upper()
    groups = []
    for end in range(len(t), 0, -k):          # bước âm: đi lùi
        groups.append(t[max(0, end - k):end])
    return "-".join(reversed(groups))

print(list(range(5, 0, -2)))                  # 5, 3, 1
print(license_slice_from_end("2-5g-3-J", 2))
print(license_slice_from_end("5F3Z-2e-9-w", 4))
print(repr(license_slice_from_end("--", 1)))

[5, 3, 1]
2-5G-3J
5F3Z-2E9W
''


`range(len(t), 0, -k)` với **bước âm** đi lùi và dừng **trước** 0. Với chuỗi rỗng, `range(0, 0, -k)` rỗng → `groups = []` → `''.join` cho `""`: ca biên tự xử lý đúng.

## 9. Code nộp LeetCode (Python 3)

In [13]:
class Solution:
    def licenseKeyFormatting(self, s: str, k: int) -> str:
        t = s.replace("-", "").upper()
        if not t:
            return ""
        first = len(t) % k or k
        groups = [t[:first]]
        for i in range(first, len(t), k):
            groups.append(t[i:i + k])
        return "-".join(groups)

sol = Solution()
print(sol.licenseKeyFormatting("5F3Z-2e-9-w", 4))   # 5F3Z-2E9W
print(sol.licenseKeyFormatting("2-5g-3-J", 2))      # 2-5G-3J

5F3Z-2E9W
2-5G-3J


## 10. Vì sao đúng?

**Cách cắt (bản nộp):** gọi `n = len(t)`.
- Nhóm đầu dài `first = n % k` nếu khác 0, ngược lại `k` → luôn `1 <= first <= k` khi `n >= 1`.
- Phần còn lại dài `n - first`, chia hết cho `k` (vì `first ≡ n (mod k)`), nên vòng `range(first, n, k)` cắt ra đúng các nhóm **đủ `k`**.
- Ghép bằng `'-'.join` → đúng một dấu `-` giữa các nhóm, không có ở đầu/cuối. Thứ tự ký tự giữ nguyên, đã viết hoa.

**Cách duyệt ngược** — invariant: sau mỗi ký tự, `out` (đọc ngược) là định dạng đúng của **hậu tố** đã duyệt, trong đó mọi nhóm đủ `k` trừ nhóm "đầu" (nhóm đang xây, `count` ký tự, `1 <= count <= k`). Ký tự mới chỉ mở nhóm mới khi nhóm hiện tại đã đủ `k`. Khi hết chuỗi, nhóm đang xây chính là nhóm đầu được phép ngắn.

## 11. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| Cộng chuỗi `+=` trong vòng lặp | `O(n²)` xấu nhất | `O(n)` | không bảo đảm nhanh |
| 1. Làm sạch + cắt (bản nộp) | `O(n)` | `O(n)` | rõ ràng nhất |
| 2. Duyệt ngược + list | `O(n)` | `O(n)` | một lượt duyệt |
| 3. Cắt từ cuối bằng slicing | `O(n)` | `O(n)` | ca rỗng tự đúng |

Bộ nhớ `O(n)` là không tránh được vì phải trả về chuỗi mới (chuỗi Python bất biến).

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual!r}")

cases = [
    ("đề 1", "5F3Z-2e-9-w", 4, "5F3Z-2E9W"),
    ("đề 2", "2-5g-3-J", 2, "2-5G-3J"),
    ("toàn dấu -", "---", 3, ""),
    ("một ký tự", "a", 2, "A"),
    ("k = 1", "a-b-c", 1, "A-B-C"),
    ("k lớn hơn n", "ab-cd", 10, "ABCD"),
    ("chia hết", "abcdef", 3, "ABC-DEF"),
    ("dấu - ở đầu/cuối", "--a-bc--", 2, "A-BC"),
    ("chỉ chữ số", "1-2-3-4-5", 2, "1-23-45"),
]
sol = Solution()
for name, s, k, expected in cases:
    check(name, sol.licenseKeyFormatting(s, k), expected)

PASS  đề 1: '5F3Z-2E9W'
PASS  đề 2: '2-5G-3J'
PASS  toàn dấu -: ''
PASS  một ký tự: 'A'
PASS  k = 1: 'A-B-C'
PASS  k lớn hơn n: 'ABCD'
PASS  chia hết: 'ABC-DEF'
PASS  dấu - ở đầu/cuối: 'A-BC'
PASS  chỉ chữ số: '1-23-45'


In [15]:
import random
random.seed(482)
alphabet = "aB3-"
for _ in range(3000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 15)))
    k = random.randint(1, 6)
    r1 = license_slice(s, k)
    r2 = license_reverse(s, k)
    r3 = license_slice_from_end(s, k)
    assert r1 == r2 == r3, (s, k, r1, r2, r3)
    # các tính chất của kết quả hợp lệ
    groups = r1.split("-") if r1 else []
    assert all(len(g) == k for g in groups[1:])
    assert not groups or 1 <= len(groups[0]) <= k
    assert r1.replace("-", "") == s.replace("-", "").upper()
print("3000 bộ ngẫu nhiên: 3 cách khớp; mọi nhóm sau đủ k, nhóm đầu 1..k, giữ nguyên ký tự")

3000 bộ ngẫu nhiên: 3 cách khớp; mọi nhóm sau đủ k, nhóm đầu 1..k, giữ nguyên ký tự


`r1.split("-")` tách chuỗi tại dấu `-` thành list — phép ngược của `'-'.join`. Lưu ý `"".split("-")` trả `['']` (một phần tử rỗng), nên ta xử lý riêng chuỗi rỗng.

## 13. Lỗi người mới hay mắc

### Lỗi 1: chia nhóm từ đầu (nhóm **cuối** bị ngắn)

In [16]:
def wrong_group_from_start(s, k):
    t = s.replace("-", "").upper()
    return "-".join(t[i:i + k] for i in range(0, len(t), k))

print("sai :", wrong_group_from_start("2-5g-3-J", 2))
print("đúng:", license_slice("2-5g-3-J", 2))

sai : 25-G3-J
đúng: 2-5G-3J


### Lỗi 2: `first = len(t) % k` mà không có `or k`
Khi chia hết, `first = 0` → nhóm đầu rỗng → kết quả bắt đầu bằng `-`.

In [17]:
def wrong_first_zero(s, k):
    t = s.replace("-", "").upper()
    first = len(t) % k              # SAI khi chia hết
    groups = [t[:first]] + [t[i:i + k] for i in range(first, len(t), k)]
    return "-".join(groups)

print("sai :", repr(wrong_first_zero("abcdef", 3)))
print("đúng:", repr(license_slice("abcdef", 3)))

sai : '-ABC-DEF'
đúng: 'ABC-DEF'


### Lỗi 3: duyệt ngược nhưng chèn `-` **sau** khi đủ nhóm
Nếu chèn ngay khi `count` đạt `k`, lúc tổng số ký tự chia hết cho `k` sẽ thừa một dấu `-` ở đầu kết quả.

In [18]:
def wrong_insert_after(s, k):
    out, count = [], 0
    for ch in reversed(s):
        if ch == "-":
            continue
        out.append(ch.upper())
        count += 1
        if count == k:              # SAI: chèn ngay sau khi đủ
            out.append("-")
            count = 0
    return "".join(reversed(out))

print("sai :", repr(wrong_insert_after("abcdef", 3)))
print("đúng:", repr(license_reverse("abcdef", 3)))

sai : '-ABC-DEF'
đúng: 'ABC-DEF'


### Lỗi 4: quên ca chuỗi toàn dấu `-`
Với cách cắt, nếu không kiểm tra `t` rỗng: `first = 0 % k or k = k`, `t[:k] = ""` → trả `""` — may mắn vẫn đúng. Nhưng nhiều cách viết khác (ví dụ `t[0]`) sẽ `IndexError`. Luôn test `"---"`.

### Lỗi 5: quên viết hoa, hoặc viết hoa cả dấu `-`
`'-'.upper() == '-'` nên viết hoa cả chuỗi không sao; nhưng quên `upper()` sẽ sai ví dụ 1 (`"2e9w"` thay vì `"2E9W"`).

## 14. Mẹo nhận dạng pattern

- "Chia thành nhóm `k`, nhóm **đầu** được lẻ" → tính từ **cuối** (duyệt ngược) hoặc `first = n % k or k`. Giống cách đặt dấu phẩy hàng nghìn: `1,234,567`.
- Ghép chuỗi trong vòng lặp → **list + `join`**.
- Bỏ ký tự rác trước (`replace`/`filter`), rồi mới xử lý cấu trúc.

Bài cùng họ: 1556 Thousand Separator, 1694 Reformat Phone Number, 68 Text Justification (Hard), 929 Unique Email Addresses, 2138 Divide a String Into Groups of Size k.

## 15. Biến thể

### 15.1. Dấu phân cách hàng nghìn (LeetCode 1556 dùng dấu `.`)

In [19]:
def thousand_separator(n: int, sep: str = ".") -> str:
    t = str(n)
    first = len(t) % 3 or 3
    return sep.join([t[:first]] + [t[i:i + 3] for i in range(first, len(t), 3)])

print(thousand_separator(987), thousand_separator(1234), thousand_separator(1234567, ","))
print("f-string có sẵn:", f"{1234567:,}")

987 1.234 1,234,567
f-string có sẵn: 1,234,567


### 15.2. Nhóm **cuối** được lẻ (chia từ đầu) — LeetCode 2138 có thêm ký tự đệm

In [20]:
def divide_string(s: str, k: int, fill: str) -> list[str]:
    groups = [s[i:i + k] for i in range(0, len(s), k)]
    groups[-1] = groups[-1] + fill * (k - len(groups[-1]))
    return groups

print(divide_string("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'jxx']


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Tính tay
`s = "r-2Ab-c9-x"`, `k = 3`. Chuỗi sạch là gì, nhóm đầu dài bao nhiêu, kết quả là gì?

### Bài 2 — Hoàn thiện cách cắt

In [21]:
def ex2_license(s: str, k: int) -> str:
    # TODO 1: t = bỏ '-' và viết hoa
    # TODO 2: nếu t rỗng -> ''
    # TODO 3: first = len(t) % k or k
    # TODO 4: groups = [t[:first]] + các nhóm k ký tự còn lại; trả '-'.join(groups)
    pass

# assert ex2_license("5F3Z-2e-9-w", 4) == "5F3Z-2E9W"
# assert ex2_license("2-5g-3-J", 2) == "2-5G-3J"

### Bài 3 — Viết lại bằng duyệt ngược (không dùng `replace`)

In [22]:
def ex3_license_reverse(s: str, k: int) -> str:
    # TODO
    pass

# assert ex3_license_reverse("2-5g-3-J", 2) == "2-5G-3J"
# assert ex3_license_reverse("---", 1) == ""

### Bài 4 — Thousand Separator cho số nguyên **có thể âm**
Ví dụ `-1234567` → `"-1.234.567"`, `0` → `"0"`.

In [23]:
def ex4_separator(n: int) -> str:
    # TODO: tách dấu âm ra trước
    pass

# assert ex4_separator(-1234567) == "-1.234.567"
# assert ex4_separator(0) == "0"
# assert ex4_separator(999) == "999"

### Bài 5 — Mini challenge: Reformat Phone Number (LeetCode 1694)
Bỏ khoảng trắng và `-`, rồi chia thành nhóm 3 từ trái sang; nhưng nếu còn lại 4 chữ số cuối thì chia `2-2` (không bao giờ có nhóm 1 chữ số). Ví dụ `"1-23-45 6"` → `"123-456"`, `"123 4-567"` → `"123-45-67"`, `"12"` → `"12"`.

In [24]:
def ex5_reformat_number(number: str) -> str:
    # TODO
    pass

# assert ex5_reformat_number("1-23-45 6") == "123-456"
# assert ex5_reformat_number("123 4-567") == "123-45-67"
# assert ex5_reformat_number("123 4-5678") == "123-456-78"
# assert ex5_reformat_number("12") == "12"

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: bỏ `-`, viết hoa. Mức 2: `n = 7`, `7 % 3 = 1`. Mức 3: `R-2AB-C9X`.

**Bài 2** — Mức 1: chép các bước ở mục 3. Mức 2: `range(first, len(t), k)`. Mức 3: nhớ `or k`.

**Bài 3** — Mức 1: `for ch in reversed(s)`. Mức 2: `if count == k:` chèn `-` **trước** khi thêm ký tự. Mức 3: `''.join(reversed(out))`.

**Bài 4** — Mức 1: `sign = "-" if n < 0 else ""`, làm việc với `abs(n)`. Mức 2: dùng `first = len(t) % 3 or 3`. Mức 3: `sign + ".".join(groups)`.

**Bài 5** — Mức 1: làm sạch bằng list comprehension `ch.isdigit()`. Mức 2: `while len(rest) > 4:` cắt 3. Mức 3: phần còn lại 4 → `rest[:2], rest[2:]`; còn 2 hoặc 3 → giữ nguyên.

## 17. Đáp án tham khảo

**Bài 1:** chuỗi sạch `"R2ABC9X"` (n = 7), nhóm đầu dài `7 % 3 = 1` → `"R-2AB-C9X"`.

In [25]:
def ans2_license(s, k):
    t = s.replace("-", "").upper()
    if not t:
        return ""
    first = len(t) % k or k
    return "-".join([t[:first]] + [t[i:i + k] for i in range(first, len(t), k)])

def ans3_license_reverse(s, k):
    out, count = [], 0
    for ch in reversed(s):
        if ch == "-":
            continue
        if count == k:
            out.append("-")
            count = 0
        out.append(ch.upper())
        count += 1
    return "".join(reversed(out))

def ans4_separator(n):
    sign = "-" if n < 0 else ""
    t = str(abs(n))
    first = len(t) % 3 or 3
    return sign + ".".join([t[:first]] + [t[i:i + 3] for i in range(first, len(t), 3)])

def ans5_reformat_number(number):
    digits = "".join(ch for ch in number if ch.isdigit())
    groups = []
    i = 0
    while len(digits) - i > 4:
        groups.append(digits[i:i + 3])
        i += 3
    rest = digits[i:]
    if len(rest) == 4:
        groups += [rest[:2], rest[2:]]
    else:
        groups.append(rest)
    return "-".join(groups)

assert ans2_license("r-2Ab-c9-x", 3) == "R-2AB-C9X"                 # bài 1
assert ans2_license("5F3Z-2e-9-w", 4) == "5F3Z-2E9W"
assert ans2_license("2-5g-3-J", 2) == "2-5G-3J"
assert ans3_license_reverse("2-5g-3-J", 2) == "2-5G-3J"
assert ans3_license_reverse("---", 1) == ""
assert ans4_separator(-1234567) == "-1.234.567"
assert ans4_separator(0) == "0" and ans4_separator(999) == "999"
assert ans5_reformat_number("1-23-45 6") == "123-456"
assert ans5_reformat_number("123 4-567") == "123-45-67"
assert ans5_reformat_number("123 4-5678") == "123-456-78"
assert ans5_reformat_number("12") == "12"
print("Đáp án bài 1–5: PASS")

Đáp án bài 1–5: PASS


`ch.isdigit()` trả `True` nếu ký tự là chữ số. `groups += [a, b]` nối thêm hai phần tử vào list (tương đương `groups.extend([a, b])`).

## 18. Template nhớ nhanh + checklist submit

```python
t = s.replace("-", "").upper()
if not t:
    return ""
first = len(t) % k or k                     # nhóm đầu: 1..k ký tự
groups = [t[:first]] + [t[i:i + k] for i in range(first, len(t), k)]
return "-".join(groups)
```

**Checklist trước khi submit**
- [ ] Đã bỏ **mọi** dấu `-` cũ và viết hoa?
- [ ] Nhóm **đầu** (không phải nhóm cuối) là nhóm được ngắn?
- [ ] Có `or k` khi số ký tự chia hết cho `k`?
- [ ] Ca toàn dấu `-` trả `""`, ca một ký tự, ca `k > n`?
- [ ] Ghép bằng list + `join`, không cộng chuỗi trong vòng lặp lớn?